# GENOMIRROR AI
## Evidence-Traceable Biomedical Knowledge Graph & Scientific Disagreement Explorer

**Project type:** Bioinformatics + knowledge graphs + evidence quality analysis  
**Notebook goal:** Build a reproducible prototype that connects genes, variants, diseases, and evidence records; flags disagreements in a *demonstration dataset*; identifies incomplete evidence records; and creates a graph visualization and exportable report.

### Important scope note
This notebook uses **synthetic demonstration data**, not real patient records or verified clinical assertions. The example classifications are invented to demonstrate the workflow. This is a research/portfolio prototype, **not a clinical decision-support tool**. A production version would require carefully sourced public records, normalized identifiers, version/date tracking, and expert validation.

## 1. What makes this project different?

Rather than only predicting a label, GENOMIRROR AI focuses on **where a claim came from and how evidence records relate**.

This prototype demonstrates:
1. A small evidence table with source and date fields.
2. A knowledge graph linking variants, genes, diseases, and evidence records.
3. A disagreement detector that flags variants with differing example classifications.
4. An evidence completeness check.
5. A visual network and machine-readable CSV/JSON exports.

**Interpretation rule:** A disagreement flag is a prompt for review—not proof that a source is wrong. Different records may use different criteria, dates, or contexts.

In [ ]:
# Install dependencies if needed (uncomment in a fresh environment)
# %pip install pandas numpy matplotlib networkx

## 2. Import libraries

In [ ]:
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 50)

OUTPUT_DIR = Path("genomirror_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Libraries loaded.")
print("Output directory:", OUTPUT_DIR.resolve())

## 3. Create a demonstration evidence dataset

In [ ]:
# Entirely synthetic records for demonstrating the software workflow.
# Do not interpret these rows as real biological or clinical evidence.
records = [
    {
        "record_id": "EV-001", "variant_id": "VAR-DEMO-001", "gene_id": "GENE-DEMO-A",
        "disease_id": "DISEASE-DEMO-X", "classification": "Pathogenic",
        "evidence_type": "functional_study", "source_name": "Synthetic Source Alpha",
        "source_url": "https://example.org/alpha", "publication_year": 2022,
        "review_status": "reviewed", "evidence_summary": "Synthetic functional evidence example."
    },
    {
        "record_id": "EV-002", "variant_id": "VAR-DEMO-001", "gene_id": "GENE-DEMO-A",
        "disease_id": "DISEASE-DEMO-X", "classification": "Uncertain significance",
        "evidence_type": "case_series", "source_name": "Synthetic Source Beta",
        "source_url": "https://example.org/beta", "publication_year": 2024,
        "review_status": "needs_review", "evidence_summary": "Synthetic record with a different example classification."
    },
    {
        "record_id": "EV-003", "variant_id": "VAR-DEMO-002", "gene_id": "GENE-DEMO-B",
        "disease_id": "DISEASE-DEMO-Y", "classification": "Benign",
        "evidence_type": "population_frequency", "source_name": "Synthetic Source Gamma",
        "source_url": "https://example.org/gamma", "publication_year": 2023,
        "review_status": "reviewed", "evidence_summary": "Synthetic population-frequency evidence example."
    },
    {
        "record_id": "EV-004", "variant_id": "VAR-DEMO-003", "gene_id": "GENE-DEMO-C",
        "disease_id": "DISEASE-DEMO-Z", "classification": "Uncertain significance",
        "evidence_type": None, "source_name": "Synthetic Source Delta",
        "source_url": None, "publication_year": None,
        "review_status": "needs_review", "evidence_summary": None
    },
    {
        "record_id": "EV-005", "variant_id": "VAR-DEMO-002", "gene_id": "GENE-DEMO-B",
        "disease_id": "DISEASE-DEMO-Y", "classification": "Benign",
        "evidence_type": "computational_prediction", "source_name": "Synthetic Source Epsilon",
        "source_url": "https://example.org/epsilon", "publication_year": 2025,
        "review_status": "reviewed", "evidence_summary": "Second synthetic record for the same example variant."
    },
]

df = pd.DataFrame(records)
df

## 4. Validate and profile the data

In [ ]:
required_columns = [
    "record_id", "variant_id", "gene_id", "disease_id", "classification",
    "source_name", "review_status"
]
missing_required = [col for col in required_columns if col not in df.columns]
if missing_required:
    raise ValueError(f"Required columns missing: {missing_required}")

profile = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "missing_count": [int(df[c].isna().sum()) for c in df.columns],
    "missing_percent": [round(float(df[c].isna().mean() * 100), 1) for c in df.columns],
    "unique_values": [int(df[c].nunique(dropna=True)) for c in df.columns],
})
display(profile)

print(f"Rows: {len(df)} | Columns: {len(df.columns)}")
print("Duplicate record IDs:", int(df["record_id"].duplicated().sum()))

## 5. Build the biomedical knowledge graph

In [ ]:
G = nx.Graph()

for _, row in df.iterrows():
    variant = f"variant:{row['variant_id']}"
    gene = f"gene:{row['gene_id']}"
    disease = f"disease:{row['disease_id']}"
    evidence = f"evidence:{row['record_id']}"

    G.add_node(variant, node_type="variant", label=row["variant_id"])
    G.add_node(gene, node_type="gene", label=row["gene_id"])
    G.add_node(disease, node_type="disease", label=row["disease_id"])
    G.add_node(evidence, node_type="evidence", label=row["record_id"],
               classification=row["classification"], source=row["source_name"],
               year=None if pd.isna(row["publication_year"]) else int(row["publication_year"]))

    G.add_edge(variant, gene, relation="associated_with_gene")
    G.add_edge(variant, disease, relation="associated_with_disease")
    G.add_edge(evidence, variant, relation="supports_or_reports")
    G.add_edge(evidence, gene, relation="mentions_gene")
    G.add_edge(evidence, disease, relation="mentions_disease")

print(f"Graph nodes: {G.number_of_nodes()}")
print(f"Graph edges: {G.number_of_edges()}")
print("Node types:", pd.Series([d["node_type"] for _, d in G.nodes(data=True)]).value_counts().to_dict())

## 6. Detect classification disagreements

In [ ]:
# This is a simple rule-based screen over the synthetic rows.
# It does not adjudicate which classification is correct.
disagreement_rows = []

for variant_id, group in df.groupby("variant_id"):
    labels = sorted(group["classification"].dropna().astype(str).unique())
    if len(labels) > 1:
        disagreement_rows.append({
            "variant_id": variant_id,
            "classification_count": len(labels),
            "classifications_seen": " | ".join(labels),
            "record_ids": " | ".join(group["record_id"].astype(str)),
            "source_names": " | ".join(group["source_name"].astype(str)),
            "years_seen": " | ".join(
                str(int(y)) if pd.notna(y) else "year_missing"
                for y in group["publication_year"]
            ),
            "recommended_next_step": "Review original source records, dates, criteria, and context."
        })

disagreements = pd.DataFrame(disagreement_rows)
if disagreements.empty:
    print("No differing classifications found in this dataset.")
else:
    display(disagreements)

## 7. Find incomplete evidence records

In [ ]:
fields_to_check = [
    "source_url", "publication_year", "evidence_type",
    "evidence_summary", "review_status"
]

completeness = df[["record_id", "variant_id", "source_name"] + fields_to_check].copy()
completeness["missing_field_count"] = completeness[fields_to_check].isna().sum(axis=1)
completeness["completeness_percent"] = (
    (1 - completeness["missing_field_count"] / len(fields_to_check)) * 100
).round(1)

incomplete_records = completeness[completeness["missing_field_count"] > 0].sort_values(
    "missing_field_count", ascending=False
)
display(incomplete_records)

## 8. Visualize the knowledge graph

In [ ]:
plt.figure(figsize=(14, 9))
pos = nx.spring_layout(G, seed=42, k=0.8)

node_types = nx.get_node_attributes(G, "node_type")
type_groups = {}
for node, node_type in node_types.items():
    type_groups.setdefault(node_type, []).append(node)

# Use Matplotlib defaults; separate node types by marker shape for readability.
markers = {"variant": "o", "gene": "s", "disease": "^", "evidence": "D"}
for node_type, nodes in type_groups.items():
    nx.draw_networkx_nodes(
        G, pos, nodelist=nodes, node_shape=markers.get(node_type, "o"),
        node_size=900, alpha=0.9, label=node_type
    )

labels = {node: data.get("label", node) for node, data in G.nodes(data=True)}
nx.draw_networkx_labels(G, pos, labels=labels, font_size=8)
nx.draw_networkx_edges(G, pos, alpha=0.35, width=1.2)

plt.title("GENOMIRROR AI — Synthetic Biomedical Evidence Graph")
plt.legend()
plt.axis("off")
plt.tight_layout()
graph_path = OUTPUT_DIR / "genomirror_knowledge_graph.png"
plt.savefig(graph_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved graph image:", graph_path)

## 9. Generate a review queue

In [ ]:
# Combine disagreement flags and incomplete records into a lightweight review queue.
review_items = []

for _, row in disagreements.iterrows():
    review_items.append({
        "priority": "High",
        "item_type": "classification_disagreement",
        "entity_id": row["variant_id"],
        "finding": f"Differing example classifications: {row['classifications_seen']}",
        "next_step": row["recommended_next_step"]
    })

for _, row in incomplete_records.iterrows():
    review_items.append({
        "priority": "Medium",
        "item_type": "incomplete_evidence_record",
        "entity_id": row["record_id"],
        "finding": f"{int(row['missing_field_count'])} key field(s) missing",
        "next_step": "Locate and verify the original source and fill missing metadata where possible."
    })

review_queue = pd.DataFrame(review_items)
if review_queue.empty:
    print("No review items generated.")
else:
    display(review_queue)

## 10. Export results

In [ ]:
# Export the synthetic dataset and analysis outputs.
df.to_csv(OUTPUT_DIR / "synthetic_evidence_records.csv", index=False)
profile.to_csv(OUTPUT_DIR / "data_quality_profile.csv", index=False)
disagreements.to_csv(OUTPUT_DIR / "classification_disagreements.csv", index=False)
incomplete_records.to_csv(OUTPUT_DIR / "incomplete_evidence_records.csv", index=False)
review_queue.to_csv(OUTPUT_DIR / "review_queue.csv", index=False)

report = {
    "project": "GENOMIRROR AI",
    "generated_at_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "dataset_note": "Synthetic demonstration data only; not real clinical evidence.",
    "row_count": int(len(df)),
    "graph_node_count": int(G.number_of_nodes()),
    "graph_edge_count": int(G.number_of_edges()),
    "disagreement_count": int(len(disagreements)),
    "incomplete_record_count": int(len(incomplete_records)),
    "review_queue_count": int(len(review_queue)),
    "limitations": [
        "The example records and classifications are synthetic.",
        "Disagreement detection is rule-based and does not decide which record is correct.",
        "No clinical use is supported.",
        "Real deployment requires source validation, identifier normalization, provenance, and expert review."
    ]
}
with open(OUTPUT_DIR / "summary_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))
print("\nFiles created:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print("-", path.name)

## 11. Next steps for a real research-grade version

1. Replace synthetic rows with properly sourced public records from resources such as ClinVar, NCBI Gene, and Ensembl.
2. Store accession IDs, retrieval dates, version information, and source URLs for each assertion.
3. Normalize variant representations and gene identifiers before comparing records.
4. Preserve the original classification and criteria from every source.
5. Create a curated benchmark of known agreements and disagreements.
6. Evaluate precision and recall for disagreement flags and completeness checks.
7. Add authentication, audit logging, tests, and robust error handling before deployment.
8. Have domain experts review outputs before making biological interpretations.

### Suggested extension
Add a **time-aware evidence view** that shows how recorded assertions change across source versions or publication dates. The timeline should distinguish a genuine change in classification from records that simply use different criteria.

---
**End of notebook.** This is a working educational prototype and a starting point for a larger system.